# Compute Volatility Change

This notebook computes the change in stock return volatility around each earnings-call event.

Following the project methodology, volatility change is defined as:

\[
\Delta Volatility = Volatility[+1,+10] - Volatility[-10,-1]
\]

where volatility is the standard deviation of daily stock returns.

For each event, we compute:
- pre-event volatility over `t = -10` to `t = -1`
- post-event volatility over `t = +1` to `t = +10`

The output is an event-level dataset containing the volatility change measure used in later regression analysis.

In [1]:
import pandas as pd
from pathlib import Path

## 1. Define file paths

In [2]:
event_ar_path = Path("../data/processed/event_abnormal_returns.csv")

output_path = Path("../data/processed/event_volatility_change.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

## 2. Load dataset

We use the event-time dataset with returns and abnormal returns.
For volatility, we use the stock return series.

In [3]:
event_ar = pd.read_csv(event_ar_path)

print("event_ar shape:", event_ar.shape)
display(event_ar.head())

event_ar shape: (24628, 16)


,event_id,ticker,file_name,call_datetime_et,after_market_close_et,event_trading_day_final,Date,t,Adj Close,index_adj_close,return,market_return,alpha,beta,expected_return,abnormal_return
0,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-05,-120,25.767361,5139.939941,0.006629,0.006736,-0.000387,1.102994,0.007043,-0.000414
1,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-06,-119,25.823441,5056.439941,0.002176,-0.016245,-0.000387,1.102994,-0.018305,0.020482
2,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-07,-118,25.910908,5043.540039,0.003387,-0.002551,-0.000387,1.102994,-0.003201,0.006588
3,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-10,-117,26.852962,5101.799805,0.036357,0.011551,-0.000387,1.102994,0.012354,0.024003
4,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-11,-116,25.455584,5036.790039,-0.052038,-0.012743,-0.000387,1.102994,-0.014442,-0.037597


## 3. Parse dates and sort data

In [4]:
event_ar["Date"] = pd.to_datetime(event_ar["Date"], errors="coerce")
event_ar["event_trading_day_final"] = pd.to_datetime(
    event_ar["event_trading_day_final"], errors="coerce"
)

event_ar = event_ar.sort_values(["event_id", "t"]).reset_index(drop=True)

## 4. Define pre-event and post-event windows

According to the methodology:
- pre-event window: `[-10,-1]`
- post-event window: `[+1,+10]`

In [5]:
pre_window = list(range(-10, 0))    # -10 ... -1
post_window = list(range(1, 11))    # 1 ... 10

print("Pre-event window:", pre_window)
print("Post-event window:", post_window)

Pre-event window: [-10, -9, -8, -7, -6, -5, -4, -3, -2, -1]
Post-event window: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


## 5. Validate coverage for each event

We confirm that every event has full pre-event and post-event volatility windows.

In [6]:
event_t_sets = event_ar.groupby("event_id")["t"].apply(set)

pre_valid = event_t_sets.apply(lambda s: set(pre_window).issubset(s))
post_valid = event_t_sets.apply(lambda s: set(post_window).issubset(s))

print("Events with valid pre-vol window:", pre_valid.sum())
print("Events with valid post-vol window:", post_valid.sum())

Events with valid pre-vol window: 188
Events with valid post-vol window: 188


## 6. Compute pre-event volatility

Volatility is the standard deviation of stock returns over `t = -10` to `t = -1`.

In [9]:
pre_vol = (
    event_ar[event_ar["t"].isin(pre_window)]
    .groupby("event_id")["return"]
    .std()
    .reset_index(name="pre_volatility")
)

## 7. Compute post-event volatility

Volatility is the standard deviation of stock returns over `t = +1` to `t = +10`.

In [7]:
post_vol = (
    event_ar[event_ar["t"].isin(post_window)]
    .groupby("event_id")["return"]
    .std()
    .reset_index(name="post_volatility")
)

## 8. Merge pre-event and post-event volatility

In [10]:
vol_df = pre_vol.merge(post_vol, on="event_id", how="inner")

print("vol_df shape:", vol_df.shape)
display(vol_df.head())

vol_df shape: (188, 3)


,event_id,pre_volatility,post_volatility
0,1,0.024005,0.018419
1,2,0.010551,0.012455
2,3,0.009706,0.010494
3,4,0.004998,0.009293
4,5,0.005383,0.004957


## 9. Compute volatility change

\[
\Delta Volatility = Volatility[+1,+10] - Volatility[-10,-1]
\]

In [11]:
vol_df["volatility_change"] = (
    vol_df["post_volatility"] - vol_df["pre_volatility"]
)

## 10. Attach event-level metadata

In [12]:
event_info = (
    event_ar[["event_id", "ticker", "event_trading_day_final"]]
    .drop_duplicates()
)

vol_df = vol_df.merge(event_info, on="event_id", how="left")
display(vol_df.head())

,event_id,pre_volatility,post_volatility,volatility_change,ticker,event_trading_day_final
0,1,0.024005,0.018419,-0.005586,AAPL,2016-01-27
1,2,0.010551,0.012455,0.001904,AAPL,2016-04-27
2,3,0.009706,0.010494,0.000788,AAPL,2016-07-27
3,4,0.004998,0.009293,0.004296,AAPL,2016-10-26
4,5,0.005383,0.004957,-0.000426,AAPL,2017-02-01


## 11. Validate results

In [13]:
print("Missing pre_volatility:", vol_df["pre_volatility"].isna().sum())
print("Missing post_volatility:", vol_df["post_volatility"].isna().sum())
print("Missing volatility_change:", vol_df["volatility_change"].isna().sum())

assert vol_df["pre_volatility"].notna().all(), "Missing pre-event volatility values"
assert vol_df["post_volatility"].notna().all(), "Missing post-event volatility values"
assert vol_df["volatility_change"].notna().all(), "Missing volatility change values"

Missing pre_volatility: 0
Missing post_volatility: 0
Missing volatility_change: 0


## 12. Inspect volatility distribution

In [14]:
display(
    vol_df[["pre_volatility", "post_volatility", "volatility_change"]].describe()
)

,pre_volatility,post_volatility,volatility_change
count,188.000000,188.000000,188.000000
mean,0.017705,0.019148,0.001443
std,0.012354,0.012383,0.010493
min,0.003988,0.004334,-0.050167
25%,0.009160,0.010363,-0.004018
50%,0.014810,0.015788,0.000651
75%,0.022364,0.024684,0.006413
max,0.104852,0.066275,0.048493


## 13. Check extreme volatility changes

Large changes are possible around earnings events, but should still be inspected.

In [15]:
extreme_vol = vol_df[vol_df["volatility_change"].abs() > 0.2].copy()

print("Extreme volatility change observations:", len(extreme_vol))
display(extreme_vol.head(20))

Extreme volatility change observations: 0


,event_id,pre_volatility,post_volatility,volatility_change,ticker,event_trading_day_final


## 14. Save output

In [16]:
vol_df = vol_df.sort_values("event_id").reset_index(drop=True)

vol_df.to_csv(output_path, index=False)

print("Saved:")
print(output_path)

Saved:
../data/processed/event_volatility_change.csv


## 15. Validation summary

This confirms:
- number of events
- coverage of volatility windows
- missing values
- extreme volatility change observations

In [17]:
summary = {
    "n_events": int(vol_df["event_id"].nunique()),
    "n_rows": len(vol_df),
    "valid_pre_window_events": int(pre_valid.sum()),
    "valid_post_window_events": int(post_valid.sum()),
    "missing_pre_volatility": int(vol_df["pre_volatility"].isna().sum()),
    "missing_post_volatility": int(vol_df["post_volatility"].isna().sum()),
    "missing_volatility_change": int(vol_df["volatility_change"].isna().sum()),
    "extreme_volatility_change_count": int(len(extreme_vol)),
}

summary_df = pd.DataFrame(summary.items(), columns=["check", "value"])
display(summary_df)

,check,value
0,n_events,188
1,n_rows,188
2,valid_pre_window_events,188
3,valid_post_window_events,188
4,missing_pre_volatility,0
5,missing_post_volatility,0
6,missing_volatility_change,0
7,extreme_volatility_change_count,0


## Conclusion

This notebook computed event-level volatility measures for each earnings call:

- pre-event volatility over `[-10,-1]`
- post-event volatility over `[+1,+10]`
- volatility change as the difference between the two

This variable is now ready to be used as one of the main dependent variables in the final regression analysis, alongside cumulative abnormal returns.